# ChromaDB Vector Store Explorer

Visualise et explore le contenu de la base de données ChromaDB contenant les documents Terraform.

In [2]:
from pathlib import Path
import pandas as pd
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
import sys

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))
from terraform_agent.config import Config

In [6]:
# Initialize configuration
config = Config(Path.cwd().parent)
vectorstore_path = config.PROJECT_ROOT / "notebooks" / ".vectorstore"

print(f"📁 Looking for vectorstore at: {vectorstore_path}")
print(f"✓ Exists: {vectorstore_path.exists()}")

📁 Looking for vectorstore at: /Users/melkouhen/audit-tools/test-langchain/notebooks/.vectorstore
✓ Exists: True


In [7]:
# Load vectorstore
embedding_function = OllamaEmbeddings(model=config.EMBEDDING_MODEL)
vectorstore = Chroma(
    collection_name="terraform_docs",
    embedding_function=embedding_function,
    persist_directory=str(vectorstore_path),
)

print("✓ Vectorstore loaded successfully")

✓ Vectorstore loaded successfully


In [8]:
# Get all documents
all_data = vectorstore.get()
ids = all_data["ids"]
metadatas = all_data["metadatas"]
documents = all_data["documents"]

print(f"📊 Total documents indexed: {len(ids)}")

📊 Total documents indexed: 9


In [9]:
# Create DataFrame for better visualization
data = []
for i, (doc_id, metadata, content) in enumerate(zip(ids, metadatas, documents)):
    data.append({
        'ID': doc_id,
        'Source': metadata.get('source', 'unknown'),
        'Content Preview': content[:80] + '...' if len(content) > 80 else content,
        'Length': len(content),
    })

df = pd.DataFrame(data)
print(df.to_string())

                                     ID                                                                Source                                                                          Content Preview  Length
0  5373ab5b-eb5d-4715-87ec-ae01f934e23d         /Users/melkouhen/audit-tools/test-langchain/docs/structure.md    # Bonnes Pratiques de Structuration Terraform\n\n## 1. Organisation Générale du Pr...     800
1  ad0e715d-d24e-438e-8880-b29de1b759cb         /Users/melkouhen/audit-tools/test-langchain/docs/structure.md  ## 3. Gestion des Environnements\n\n| Contexte | Règle |\n|----------|-------|\n| **...     885
2  4b954fa8-7956-460e-9087-68364c2f903e         /Users/melkouhen/audit-tools/test-langchain/docs/structure.md  ## 5. État et Verrouillage\n\n| Contexte | Règle |\n|----------|-------|\n| **tf-rem...     981
3  556e2b1b-0207-402f-b659-859af473131f         /Users/melkouhen/audit-tools/test-langchain/docs/structure.md  ## 7. Sécurité et Secrets\n\n| Contexte | Règle |\n|---------

In [9]:
# Group by source
print("\n📚 Documents by Source:")
print(df.groupby('Source').size())


📚 Documents by Source:
Source
/Users/melkouhen/audit-tools/test-langchain/docs/cloud-storage.md                 16
/Users/melkouhen/audit-tools/test-langchain/docs/structure.md                      4
/Users/melkouhen/audit-tools/test-langchain/notebooks/../docs/cloud-storage.md    69
/Users/melkouhen/audit-tools/test-langchain/notebooks/../docs/structure.md        28
dtype: int64


In [10]:
# Content length statistics
print("\n📈 Content Length Statistics:")
print(df['Length'].describe())


📈 Content Length Statistics:
count    117.000000
mean     841.059829
std      197.403634
min      138.000000
25%      819.000000
50%      927.000000
75%      982.000000
max      999.000000
Name: Length, dtype: float64


## Search Documents

Recherche sémantique dans la base de données

In [10]:
# Interactive search
query = "module cloud storage"
k = 5

results = vectorstore.similarity_search(query, k=k)

print(f"🔍 Search Results for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [Source: {result.metadata.get('source', 'unknown')}]")
    print(f"   {result.page_content[:150]}...\n")

🔍 Search Results for: 'module cloud storage'

1. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/terraform-module.md]
   # Bonne Pratique : Utilisation de Modules Terraform

## Module : Terraform Google Cloud Storage

| Contexte | Règle |
|----------|-------|
| **tf-gcs-...

2. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/terraform-module.md]
   ### Exemple d'Usage

```hcl
module "gcs_buckets" {
  source  = "terraform-google-modules/cloud-storage/google"
  version = "~> 12.3"
  
  project_id =...

3. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/terraform-module.md]
   ### Permissions Requises

Service account ou utilisateur doit avoir le rôle : `roles/storage.admin`

### APIs Requises

Le projet doit avoir activée :...

4. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/structure.md]
   # Bonnes Pratiques de Structuration Terraform

## 1. Organisation Générale du Projet

| Contexte | Règle |
|----------|-------|
| **tf-project-layout*...

5

In [ ]:
# Advanced: Find documents about specific topics
topics = [
    "security",
    "modules",
    "variables",
    "state"
]

print("\n🎯 Topic-based Search Results:\n")
for topic in topics:
    results = vectorstore.similarity_search(topic, k=2)
    print(f"\n📌 {topic.upper()}:")
    if results:
        print(f"   {results[0].page_content[:150]}...")
    else:
        print("   No results found")

In [ ]:
# Summary
print("\n" + "="*60)
print("📊 VECTORSTORE SUMMARY")
print("="*60)
print(f"Total chunks indexed: {len(ids)}")
print(f"Unique source files: {df['Source'].nunique()}")
print(f"Average chunk size: {df['Length'].mean():.0f} characters")
print(f"Embedding model: {config.EMBEDDING_MODEL}")
print(f"Collection: terraform_docs")